In [0]:
%pip install mistralai

In [0]:
dbutils.library.restartPython()

In [0]:
from mistralai import Mistral
import json

client = Mistral(api_key="mu5qrdIw3EjA9gimrCUlN7VRyppzyopV")

test_text = "Patient has allergic rhinitis and asthma. Prescribed Zyrtec and Nasonex nasal spray."

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {
            "role": "system",
            "content": """You are a medical NLP assistant.
Extract the following and return ONLY valid JSON no backticks:
{
  "diagnosis": "primary diagnosis",
  "medications": ["list of medications"],
  "symptoms": ["list of symptoms"],
  "summary": "one sentence summary"
}"""
        },
        {
            "role": "user",
            "content": test_text
        }
    ],
    temperature=0.1,
    max_tokens=512
)

print(response.choices[0].message.content)

In [0]:
from pyspark.sql.functions import pandas_udf, col, current_timestamp
from pyspark.sql.types import StringType
import pandas as pd
import json

MISTRAL_API_KEY = ""

@pandas_udf(StringType())
def extract_medical_entities(transcriptions: pd.Series) -> pd.Series:
    from mistralai import Mistral
    client = Mistral(api_key=MISTRAL_API_KEY)
    
    results = []
    for transcription in transcriptions:
        if not transcription or len(transcription.strip()) == 0:
            results.append(json.dumps({"error": "empty"}))
            continue
        try:
            response = client.chat.complete(
                model="mistral-small-latest",
                messages=[
                    {
                        "role": "system",
                        "content": """You are a medical NLP assistant.
Extract the following and return ONLY valid JSON no backticks:
{
  "diagnosis": "primary diagnosis",
  "medications": ["list of medications"],
  "symptoms": ["list of symptoms"],
  "summary": "one sentence summary"
}"""
                    },
                    {
                        "role": "user",
                        "content": transcription[:3000]
                    }
                ],
                temperature=0.1,
                max_tokens=512
            )
            results.append(response.choices[0].message.content)
        except Exception as e:
            results.append(json.dumps({"error": str(e)}))
    
    return pd.Series(results)

In [0]:
df_silver = spark.table("medical_catalog.silver.medical_transcriptions")

df_enriched = df_silver.limit(500) \
    .withColumn("llm_raw_output", extract_medical_entities(col("transcription"))) \
    .withColumn("llm_processed_at", current_timestamp())

display(df_enriched.select("no", "medical_specialty", "llm_raw_output").limit(10))

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.sql.functions import from_json, udf

def clean_llm_output(raw):
    if not raw:
        return raw
    cleaned = raw.strip()
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()
    return cleaned

clean_udf = udf(clean_llm_output, StringType())

llm_schema = StructType([
    StructField("diagnosis",   StringType(), True),
    StructField("medications", ArrayType(StringType()), True),
    StructField("symptoms",    ArrayType(StringType()), True),
    StructField("summary",     StringType(), True),
    StructField("error",       StringType(), True)
])

df_enriched_parsed = df_enriched \
    .withColumn("llm_cleaned",     clean_udf(col("llm_raw_output"))) \
    .withColumn("llm_parsed",      from_json(col("llm_cleaned"), llm_schema)) \
    .withColumn("llm_diagnosis",   col("llm_parsed.diagnosis")) \
    .withColumn("llm_medications", col("llm_parsed.medications")) \
    .withColumn("llm_symptoms",    col("llm_parsed.symptoms")) \
    .withColumn("llm_summary",     col("llm_parsed.summary")) \
    .drop("llm_parsed", "llm_cleaned", "llm_raw_output")

print("Parsed rows:", df_enriched_parsed.count())
display(df_enriched_parsed.select(
    "no", "medical_specialty",
    "llm_diagnosis", "llm_medications",
    "llm_symptoms", "llm_summary"
).limit(10))

In [0]:
df_enriched_parsed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.silver.medical_transcriptions_enriched_temp")

print("✅ Temp table saved!")

In [0]:
df_final = spark.table("medical_catalog.silver.medical_transcriptions_enriched_temp")

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medical_catalog.silver.medical_transcriptions_enriched")

print("Enriched table saved!")
print("Row count:", spark.table("medical_catalog.silver.medical_transcriptions_enriched").count())

In [0]:
# Verify secret is loaded
key = dbutils.secrets.get(scope="medical_project", key="mistral_api_key")
print("Key loaded:", key is not None)
# Output should be: Key loaded: True